# ONNX Export: NJ Housing Price Predictor

Merges LoRA adapter into base model at fp32 and exports to ONNX.

## What this notebook does
1. Mounts Google Drive and verifies adapter exists
2. Reloads Qwen2.5-0.5B in fp32 on CPU (no quantization)
3. Loads LoRA adapter and calls merge_and_unload()
4. Saves merged fp32 model to Drive
5. Exports to ONNX via HuggingFace Optimum
6. Validates ONNX output against PyTorch at atol=1e-3

## Requirements
- Google Colab (GPU optional -- merge and export run on CPU)
- Trained LoRA adapter on Google Drive at housing_model/lora_adapter/
- ~4GB free disk space for merged model + ONNX output

## Critical Pattern
The base model MUST be reloaded in fp32 (no load_in_4bit).
Merging while still quantized produces corrupted weights silently.

In [ ]:
import os
import time

from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/housing_model"
DRIVE_ADAPTER = os.path.join(DRIVE_BASE, "lora_adapter")
DRIVE_MERGED = os.path.join(DRIVE_BASE, "merged_model")
DRIVE_ONNX = os.path.join(DRIVE_BASE, "onnx_model")

os.makedirs(DRIVE_MERGED, exist_ok=True)
os.makedirs(DRIVE_ONNX, exist_ok=True)

# Verify adapter exists
assert os.path.isdir(DRIVE_ADAPTER), f"Adapter not found at {DRIVE_ADAPTER}. Run 02_train.ipynb first."
assert os.path.isfile(os.path.join(DRIVE_ADAPTER, "adapter_config.json")), "adapter_config.json missing"

print(f"Drive paths configured:")
print(f"  Adapter:      {DRIVE_ADAPTER}")
print(f"  Merged model: {DRIVE_MERGED}")
print(f"  ONNX output:  {DRIVE_ONNX}")
print(f"Adapter found on Drive.")

_start_time = time.time()

In [ ]:
%%capture install_output
!pip install \
    "transformers>=4.45.0,<5.0.0" \
    peft==0.18.1 \
    accelerate==1.12.0 \
    optimum==2.1.0 \
    onnxruntime==1.24.2 \
    sentencepiece==0.2.1
# Install onnx extra separately to avoid shell quoting issues
!pip install "optimum[onnxruntime]"
# NOTE: bitsandbytes is intentionally NOT installed.
# The merge step reloads the base model in fp32 -- no quantization needed.
# NOTE: transformers pinned to 4.x due to pad_token_id bug in 5.x with Qwen2

In [ ]:
print("Dependencies installed. Key packages:")
import transformers, peft, accelerate, onnxruntime
from importlib.metadata import version as pkg_version
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  accelerate:    {accelerate.__version__}")
print(f"  optimum:       {pkg_version('optimum')}")
print(f"  onnxruntime:   {onnxruntime.__version__}")
print(f"\nbitsandbytes: NOT installed (intentional -- fp32 merge needs no quantization)")

In [ ]:
import json
import numpy as np
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Using CPU for fp32 merge (GPU not required)")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"

# CRITICAL: Load in fp32 on CPU -- NO quantization
# This is the two-step merge pattern (ONNX-01). If you use load_in_4bit here,
# merge_and_unload() produces corrupted weights silently.
print(f"Loading {MODEL_ID} in fp32 on CPU...")
load_start = time.time()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,   # MUST be fp32, NOT load_in_4bit
    device_map="cpu",            # CPU merge avoids GPU VRAM pressure
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

load_elapsed = time.time() - load_start
print(f"Base model loaded in fp32 on CPU ({load_elapsed:.0f}s)")
print(f"Parameters: {base_model.num_parameters():,}")
print(f"dtype: {next(base_model.parameters()).dtype}")  # Should be torch.float32

In [ ]:
# Load LoRA adapter on top of fp32 base
print(f"Loading LoRA adapter from {DRIVE_ADAPTER}...")
model_with_adapter = PeftModel.from_pretrained(base_model, DRIVE_ADAPTER)
print("Adapter loaded.")

# Merge adapter weights into base model
# After this call, LoRA deltas are added to the base weights
# and the PeftModel wrapper is removed
print("Merging adapter weights (merge_and_unload)...")
merge_start = time.time()
merged_model = model_with_adapter.merge_and_unload()
merge_elapsed = time.time() - merge_start

print(f"Merge complete in {merge_elapsed:.1f}s")
print(f"Merged model type: {type(merged_model).__name__}")
print(f"Parameters: {merged_model.num_parameters():,}")
print(f"dtype: {next(merged_model.parameters()).dtype}")  # Should still be torch.float32

In [ ]:
print(f"Saving merged model to {DRIVE_MERGED}...")
save_start = time.time()

merged_model.save_pretrained(DRIVE_MERGED)
tokenizer.save_pretrained(DRIVE_MERGED)

save_elapsed = time.time() - save_start
print(f"Merged model saved in {save_elapsed:.0f}s")

# List saved files
saved_files = os.listdir(DRIVE_MERGED)
total_size = sum(os.path.getsize(os.path.join(DRIVE_MERGED, f)) for f in saved_files) / 1e9
print(f"Files ({len(saved_files)}, total {total_size:.2f} GB):")
for f in sorted(saved_files):
    fpath = os.path.join(DRIVE_MERGED, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

In [ ]:
# ONNX export using optimum CLI
# Task: text-generation-with-past (correct for causal LM autoregressive generation)
# NOT feature-extraction (that's for embeddings only)
# ONNX-02: Export merged fp32 model to ONNX format

print("Exporting to ONNX via optimum-cli...")
print(f"  Source: {DRIVE_MERGED}")
print(f"  Output: {DRIVE_ONNX}")
print(f"  Task: text-generation-with-past")
export_start = time.time()

!optimum-cli export onnx \
    --model "{DRIVE_MERGED}" \
    --task text-generation-with-past \
    --trust-remote-code \
    "{DRIVE_ONNX}" 2>&1

export_elapsed = time.time() - export_start
print(f"\nExport completed in {export_elapsed/60:.1f} minutes")

# List ONNX output files
if os.path.isdir(DRIVE_ONNX):
    onnx_files = os.listdir(DRIVE_ONNX)
    total_onnx_size = sum(os.path.getsize(os.path.join(DRIVE_ONNX, f)) for f in onnx_files) / 1e9
    print(f"ONNX files ({len(onnx_files)}, total {total_onnx_size:.2f} GB):")
    for f in sorted(onnx_files):
        fpath = os.path.join(DRIVE_ONNX, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f} ({size_mb:.1f} MB)")
else:
    print("ERROR: ONNX output directory not created. Export may have failed.")

In [ ]:
# FALLBACK: If the default export failed (e.g., RoPE issues with TorchScript exporter),
# try the dynamo exporter with opset 18
# Only run this cell if Cell 8 failed

import glob

onnx_model_files = glob.glob(os.path.join(DRIVE_ONNX, "*.onnx"))
if onnx_model_files:
    print(f"ONNX files already exist ({len(onnx_model_files)} .onnx files). Skipping fallback.")
else:
    print("No .onnx files found. Trying dynamo exporter...")
    !optimum-cli export onnx \
        --model "{DRIVE_MERGED}" \
        --task text-generation-with-past \
        --trust-remote-code \
        --dynamo \
        --opset 18 \
        "{DRIVE_ONNX}" 2>&1

    onnx_model_files = glob.glob(os.path.join(DRIVE_ONNX, "*.onnx"))
    if onnx_model_files:
        print(f"Dynamo export succeeded: {len(onnx_model_files)} .onnx files")
    else:
        print("ERROR: Both default and dynamo exports failed.")
        print("Manual debugging required. Check error messages above.")

In [ ]:
# ONNX-03: Validate ONNX output matches PyTorch at atol=1e-3
from optimum.onnxruntime import ORTModelForCausalLM
from datasets import load_dataset

# Load a test prompt
ds = load_dataset("rajkumar4466/nj-housing-prices")
test_prompt = ds["test"][0]["prompt"]
print(f"Validation prompt: {test_prompt[:80]}...")

# Tokenize
inputs = tokenizer(test_prompt, return_tensors="pt")

# PyTorch inference (merged fp32 model)
print("\nPyTorch forward pass...")
merged_model.eval()
with torch.no_grad():
    pt_outputs = merged_model(**{k: v for k, v in inputs.items()})
pt_logits = pt_outputs.logits[0, -1, :].numpy()  # last token logits
print(f"  PyTorch logits shape: {pt_logits.shape}")
print(f"  PyTorch logits sample (first 5): {pt_logits[:5]}")

# ONNX Runtime inference
print("\nONNX Runtime forward pass...")
ort_model = ORTModelForCausalLM.from_pretrained(DRIVE_ONNX)
with torch.no_grad():
    ort_outputs = ort_model(**inputs)
ort_logits = ort_outputs.logits[0, -1, :].numpy()
print(f"  ONNX logits shape: {ort_logits.shape}")
print(f"  ONNX logits sample (first 5): {ort_logits[:5]}")

# Compare
max_diff = np.max(np.abs(pt_logits.astype(np.float32) - ort_logits.astype(np.float32)))
mean_diff = np.mean(np.abs(pt_logits.astype(np.float32) - ort_logits.astype(np.float32)))

print(f"\n{'='*60}")
print(f"ONNX NUMERICAL VALIDATION (ONNX-03)")
print(f"{'='*60}")
print(f"Max absolute difference:  {max_diff:.6f}")
print(f"Mean absolute difference: {mean_diff:.6f}")
print(f"Tolerance (atol):         1e-3 = 0.001000")

if max_diff < 1e-3:
    print(f"\nONNX-03 PASSED: max_diff={max_diff:.6f} < atol=1e-3")
else:
    print(f"\nONNX-03 FAILED: max_diff={max_diff:.6f} > atol=1e-3")
    print("This may be due to fp16/fp32 precision mismatch.")
    print("If the export used fp16, try re-exporting with fp32 (see next cell).")
print(f"{'='*60}")

# Clean up ORT model to free memory
del ort_model

In [ ]:
# Alternative validation using raw onnxruntime session
# This is a fallback -- skip if Cell 10 (ORTModelForCausalLM) already passed
import onnxruntime as ort
import glob

onnx_files = glob.glob(os.path.join(DRIVE_ONNX, "*.onnx"))
if not onnx_files:
    print("No .onnx files found. Skipping raw validation.")
else:
    onnx_path = onnx_files[0]
    print(f"Raw ONNX Runtime validation using: {os.path.basename(onnx_path)}")

    session = ort.InferenceSession(onnx_path)
    input_names = [inp.name for inp in session.get_inputs()]
    print(f"ONNX input names: {input_names}")

    # Prepare inputs
    ort_inputs = {}
    if "input_ids" in input_names:
        ort_inputs["input_ids"] = inputs["input_ids"].numpy()
    if "attention_mask" in input_names:
        ort_inputs["attention_mask"] = inputs["attention_mask"].numpy()
    if "position_ids" in input_names:
        seq_len = inputs["input_ids"].shape[1]
        ort_inputs["position_ids"] = np.arange(seq_len, dtype=np.int64).reshape(1, -1)

    # Fill any required past_key_values with zeros
    for inp in session.get_inputs():
        if inp.name.startswith("past_key_values") and inp.name not in ort_inputs:
            shape = [s if isinstance(s, int) else 1 for s in inp.shape]
            ort_inputs[inp.name] = np.zeros(shape, dtype=np.float32)

    ort_raw_out = session.run(None, ort_inputs)
    ort_raw_logits = ort_raw_out[0][0, -1, :]

    max_diff_raw = np.max(np.abs(pt_logits.astype(np.float32) - ort_raw_logits.astype(np.float32)))
    print(f"Raw ONNX max absolute difference: {max_diff_raw:.6f}")
    if max_diff_raw < 1e-3:
        print(f"ONNX-03 PASSED (raw session): max_diff={max_diff_raw:.6f} < atol=1e-3")
    else:
        print(f"ONNX-03 result (raw session): max_diff={max_diff_raw:.6f}")

    del session

In [ ]:
# Verify the ONNX model can actually generate text (not just logits)
print("Testing end-to-end ONNX generation...")

ort_gen_model = ORTModelForCausalLM.from_pretrained(DRIVE_ONNX)

test_prompt = ds["test"][0]["prompt"]
gen_inputs = tokenizer(test_prompt, return_tensors="pt")

with torch.no_grad():
    gen_outputs = ort_gen_model.generate(
        **gen_inputs,
        max_new_tokens=20,
        do_sample=False,
    )

generated_text = tokenizer.decode(gen_outputs[0], skip_special_tokens=True)
generated_suffix = generated_text[len(test_prompt):]

print(f"Prompt: ...{test_prompt[-50:]}")
print(f"Generated: {generated_suffix[:40]}")

import re
price_match = re.search(r"\d+(?:\.\d+)?", generated_suffix.replace(",", ""))
if price_match:
    print(f"Parsed price: ${float(price_match.group()):,.0f}")
    print(f"Actual price: ${ds['test'][0]['price']:,.0f}")
    print("ONNX generation test PASSED: Model generates parseable prices.")
else:
    print("WARNING: Could not parse a price from ONNX generation output.")

del ort_gen_model

In [ ]:
total_elapsed = time.time() - _start_time

print("=" * 60)
print("ONNX EXPORT SUMMARY")
print("=" * 60)
print(f"Model:          Qwen/Qwen2.5-0.5B + QLoRA adapter")
print(f"Merge pattern:  fp32 reload on CPU -> merge_and_unload()")
print(f"Merged model:   {DRIVE_MERGED}")
print(f"ONNX output:    {DRIVE_ONNX}")
print(f"")

# List ONNX files with sizes
if os.path.isdir(DRIVE_ONNX):
    onnx_files = sorted(os.listdir(DRIVE_ONNX))
    for f in onnx_files:
        fpath = os.path.join(DRIVE_ONNX, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f} ({size_mb:.1f} MB)")

print(f"")
print(f"Total export time: {total_elapsed/60:.1f} minutes")
print("=" * 60)

# Requirement checks
print("\nRequirement Status:")
merged_exists = os.path.isdir(DRIVE_MERGED) and os.listdir(DRIVE_MERGED)
print(f"  ONNX-01 (fp32 merge): {'PASSED' if merged_exists else 'FAILED'} -- merged model at {DRIVE_MERGED}")

import glob
onnx_model_exists = len(glob.glob(os.path.join(DRIVE_ONNX, "*.onnx"))) > 0
print(f"  ONNX-02 (export):     {'PASSED' if onnx_model_exists else 'FAILED'} -- ONNX files at {DRIVE_ONNX}")

# ONNX-03 was checked in Cell 9 -- report the result
try:
    if max_diff < 1e-3:
        print(f"  ONNX-03 (validation): PASSED -- max_diff={max_diff:.6f} < atol=1e-3")
    else:
        print(f"  ONNX-03 (validation): FAILED -- max_diff={max_diff:.6f} > atol=1e-3")
except NameError:
    print(f"  ONNX-03 (validation): UNKNOWN -- max_diff variable not set (validation cell may not have run)")